# Kombinierte RBF-SVM mit Index-Katalog

**Hypothese:** Der geprüfte Vegetationsindex-Katalog ergänzt Rohspektren, AEZ und Monat mit fachlich verdichteten Merkmalen und verbessert die kombinierte Balanced Accuracy gegenüber `combined_svm_expanded`.

Der Versuch verändert nur die Merkmalsgruppe `use_vegetation_indices`. Die gemeinsame Pipeline berechnet die Indizes innerhalb jedes Trainingsfolds; die Rohspektren bleiben als Referenzmerkmale enthalten. Ein reiner Indexansatz gehört zu #124.

In [ ]:
%load_ext autoreload
%autoreload 2

import pandas as pd
from IPython.display import display

from awp2.experiment import RunConfig, TuneConfig, run, tune
from awp2.models import make_combined_svm
from awp2.preprocessing import PreprocessingConfig

EXPERIMENT_ID = "combined_svm_index_catalog"
TUNING_ID = f"{EXPERIMENT_ID}_search"
REFERENCE_CV_BACC = 0.847
REFERENCE_VAL_BACC = 0.866
PREPROCESSING = PreprocessingConfig(
    use_meta=True,
    use_cyclic_month=False,
    use_spectral_standard_scale=True,
    use_vegetation_indices=True,
)
PARAM_GRID = {
    "estimator__C": [30.0, 100.0, 300.0],
    "estimator__gamma": [0.001, 0.003, 0.01],
}

## 1. Hyperparameter auf Trainings-Folds suchen

Die neun Kombinationen werden ausschließlich auf den gemeinsamen Trainings-Folds nach kombinierter Balanced Accuracy bewertet. Der Validierungsanteil bleibt bis zur nächsten Zelle unberührt.

In [ ]:
tuned = tune(
    make_combined_svm(),
    TuneConfig(
        name=TUNING_ID,
        approach="combined",
        description=(
            "RBF-SVM mit dem vollständigen reproduzierbaren Vegetationsindex-Katalog; "
            "C und gamma werden per CV verglichen."
        ),
        preprocessing=PREPROCESSING,
        param_grid=PARAM_GRID,
    ),
)

display(tuned.candidates.sort_values("rank"))
{
    "best_cv_bacc_combined": tuned.best_score,
    "best_params": tuned.best_params,
    "tuning_run": tuned.run_id,
}

## 2. Einmal auf der Validierung bewerten

Diese Zelle erst ausführen, nachdem der CV-Gewinner feststeht. Sie bewertet genau diesen Kandidaten einmal auf dem gemeinsamen Validierungsanteil.

In [ ]:
result = run(
    tuned.best_model,
    RunConfig(
        name=EXPERIMENT_ID,
        approach="combined",
        description=(
            "CV-Gewinner der RBF-SVM mit dem vollständigen reproduzierbaren "
            "Vegetationsindex-Katalog."
        ),
        preprocessing=PREPROCESSING,
        tuning_run=tuned.run_id,
    ),
)

result.metrics.model_dump()

## 3. Gegen die Referenz einordnen

Der Vergleich stellt CV und Validierung getrennt der Referenz `combined_svm_expanded` gegenüber. Die Diagramme zeigen die Kandidatenstreuung, die Fehlerstruktur sowie die häufigsten Verwechslungen für Kultur und Stadium.

In [ ]:
from awp2.data import combined_label
from awp2.evaluation import plot_confusion_matrices
from awp2.plots import plot_class_recall, plot_top_confusions, plot_tuning_candidates

comparison = pd.DataFrame(
    {
        "CV-BAcc kombiniert": {
            "Index-Katalog": tuned.best_score,
            "RBF-SVM-Referenz": REFERENCE_CV_BACC,
        },
        "Validierungs-BAcc kombiniert": {
            "Index-Katalog": result.metrics.bacc_combined,
            "RBF-SVM-Referenz": REFERENCE_VAL_BACC,
        },
    }
)
display(comparison)
display(result.metrics.model_dump())

top_confusions = (
    pd.DataFrame(
        {
            "Wahr": combined_label(result.y_val),
            "Vorhergesagt": combined_label(result.y_pred),
        }
    )
    .query("Wahr != Vorhergesagt")
    .value_counts()
    .head(10)
)
display(top_confusions)

tuning_ax = plot_tuning_candidates(
    tuned.candidates,
    color_by="estimator__C",
    validation_score=result.metrics.bacc_combined,
)
tuning_ax.set_title("Kombinierte RBF-SVM mit Index-Katalog")
display(tuning_ax.figure)

for figure in (
    plot_confusion_matrices(result.y_val, result.y_pred),
    plot_class_recall(result.y_val, result.y_pred),
    plot_top_confusions(result.y_val, result.y_pred),
):
    display(figure)

## 4. Abbildungen für die Dokumentation sichern

Diese Zelle nur ausführen, wenn der Lauf im Experiment-Log und auf der Ansatzseite dokumentiert wird.

In [ ]:
from awp2.config import MODEL_DOCS_FIGURES_DIR
from awp2.plots import save_doc_figure

save_doc_figure(
    tuning_ax.figure,
    "combined_svm_index_catalog_tuning",
    MODEL_DOCS_FIGURES_DIR,
)
save_doc_figure(
    plot_confusion_matrices(result.y_val, result.y_pred),
    "combined_svm_index_catalog_confusion",
    MODEL_DOCS_FIGURES_DIR,
)
save_doc_figure(
    plot_class_recall(result.y_val, result.y_pred),
    "combined_svm_index_catalog_recall",
    MODEL_DOCS_FIGURES_DIR,
)
save_doc_figure(
    plot_top_confusions(result.y_val, result.y_pred),
    "combined_svm_index_catalog_confusions",
    MODEL_DOCS_FIGURES_DIR,
)
{
    "tuning_run": tuned.run_id,
    "evaluation_run": result.run_id,
    "invalid_combinations": result.metrics.invalid_combinations,
}